[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/bases-de-datos-vectoriales/03-reranking.ipynb)

# Reranking para Mejorar Resultados de Búsqueda

> Aprende a mejorar la precisión de sistemas RAG usando Cross-Encoders para reordenar
> los resultados de la búsqueda vectorial inicial y métricas NDCG y MRR.

## Objetivos
- Implementar reranking con Cross-Encoder de sentence-transformers
- Calcular NDCG y MRR para evaluar la mejora
- Construir pipeline completo: ChromaDB → Cross-Encoder → Claude
- Visualizar la mejora con matplotlib

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic chromadb sentence-transformers matplotlib --quiet

## 2. Configuración e indexación del corpus

In [ ]:
import anthropic
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
import matplotlib.pyplot as plt

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print("Cargando modelos...")
bi_encoder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Corpus de 15 documentos
CORPUS = [
    "Python es el lenguaje de programación más popular en ciencia de datos e IA.",
    "NumPy permite operaciones vectoriales y matriciales eficientes en Python.",
    "Pandas facilita el análisis y manipulación de datos tabulares en Python.",
    "El machine learning supervisado aprende de ejemplos con etiquetas.",
    "Las redes neuronales convolucionales son ideales para reconocimiento de imágenes.",
    "El gradient descent optimiza los parámetros de un modelo minimizando el error.",
    "Los LLMs generan texto prediciendo el siguiente token de forma autorregresiva.",
    "El fine-tuning especializa modelos preentrenados en tareas concretas.",
    "RAG recupera documentos relevantes para fundamentar respuestas de LLMs.",
    "Los embeddings representan texto como vectores en espacios de alta dimensión.",
    "Docker empaqueta aplicaciones con sus dependencias en contenedores portables.",
    "Kubernetes orquesta contenedores Docker en clústeres de producción.",
    "FastAPI es un framework Python asíncrono para construir APIs REST rápidamente.",
    "PostgreSQL es una base de datos relacional robusta con soporte JSON y full-text.",
    "Redis es un almacén de datos en memoria ultra-rápido usado para caché y colas.",
]

# Indexar en ChromaDB
chroma = chromadb.EphemeralClient()
col = chroma.create_collection("corpus")
col.add(
    ids=[str(i) for i in range(len(CORPUS))],
    documents=CORPUS,
    embeddings=bi_encoder.encode(CORPUS).tolist()
)
print(f"✓ {len(CORPUS)} documentos indexados.")

## 3. Reranking con Cross-Encoder

In [ ]:
def recuperar_sin_reranking(query: str, n: int = 5) -> list[str]:
    """Recuperación estándar con bi-encoder."""
    emb = bi_encoder.encode([query]).tolist()
    res = col.query(query_embeddings=emb, n_results=n)
    return res["documents"][0]

def recuperar_con_reranking(query: str, n_candidatos: int = 10, n_final: int = 5) -> list[str]:
    """Recuperación con bi-encoder + reranking con cross-encoder."""
    # Paso 1: recuperar muchos candidatos con bi-encoder (recall alto)
    emb = bi_encoder.encode([query]).tolist()
    res = col.query(query_embeddings=emb, n_results=min(n_candidatos, len(CORPUS)))
    candidatos = res["documents"][0]

    # Paso 2: reordenar con cross-encoder (precisión alta)
    pares = [(query, doc) for doc in candidatos]
    scores = cross_encoder.predict(pares)

    # Ordenar por score del cross-encoder
    ordenados = sorted(zip(candidatos, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ordenados[:n_final]]

QUERY_TEST = "¿Cómo aprenden los modelos de IA de los datos?"
print(f"Query: '{QUERY_TEST}'\n")

sin_rerank = recuperar_sin_reranking(QUERY_TEST, n=5)
con_rerank = recuperar_con_reranking(QUERY_TEST, n_candidatos=10, n_final=5)

print("=== SIN RERANKING (bi-encoder) ===")
for i, doc in enumerate(sin_rerank, 1):
    print(f"  {i}. {doc[:90]}")

print("\n=== CON RERANKING (cross-encoder) ===")
for i, doc in enumerate(con_rerank, 1):
    print(f"  {i}. {doc[:90]}")

## 4. Métricas NDCG y MRR

In [ ]:
def calcular_ndcg(resultados: list[str], relevantes: list[str], k: int = 5) -> float:
    """Calcula NDCG@k: mide si los documentos relevantes aparecen primero."""
    ganancias = [1 if doc in relevantes else 0 for doc in resultados[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(ganancias))
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevantes), k)))
    return dcg / idcg if idcg > 0 else 0

def calcular_mrr(resultados: list[str], relevantes: list[str]) -> float:
    """Calcula MRR: inverso del rango del primer resultado relevante."""
    for i, doc in enumerate(resultados, 1):
        if doc in relevantes:
            return 1 / i
    return 0

# Definir qué documentos son realmente relevantes para cada query
EVALUACION = [
    {"query": "¿Cómo aprenden los modelos de IA?",
     "relevantes": [CORPUS[3], CORPUS[5], CORPUS[6]]},
    {"query": "¿Qué frameworks Python usar para APIs?",
     "relevantes": [CORPUS[12], CORPUS[0]]},
    {"query": "¿Cómo almacenar vectores en una base de datos?",
     "relevantes": [CORPUS[9], CORPUS[13], CORPUS[14]]}
]

print("=== COMPARATIVA DE MÉTRICAS ===")
print(f"{'Query':<40} {'NDCG sin':>10} {'NDCG con':>10} {'MRR sin':>8} {'MRR con':>8}")
print("-" * 80)

ndcg_sin_list, ndcg_con_list = [], []
for ev in EVALUACION:
    sin_r = recuperar_sin_reranking(ev["query"], n=5)
    con_r = recuperar_con_reranking(ev["query"], n_candidatos=10, n_final=5)
    ndcg_sin = calcular_ndcg(sin_r, ev["relevantes"])
    ndcg_con = calcular_ndcg(con_r, ev["relevantes"])
    mrr_sin = calcular_mrr(sin_r, ev["relevantes"])
    mrr_con = calcular_mrr(con_r, ev["relevantes"])
    ndcg_sin_list.append(ndcg_sin)
    ndcg_con_list.append(ndcg_con)
    print(f"{ev['query'][:40]:<40} {ndcg_sin:>10.3f} {ndcg_con:>10.3f} {mrr_sin:>8.3f} {mrr_con:>8.3f}")

print("-" * 80)
print(f"{'MEDIA':<40} {np.mean(ndcg_sin_list):>10.3f} {np.mean(ndcg_con_list):>10.3f}")

## 5. Visualización y pipeline completo con Claude

In [ ]:
# Gráfica comparativa
fig, ax = plt.subplots(figsize=(9, 5))
queries_cortas = [ev["query"][:30] for ev in EVALUACION]
x = np.arange(len(queries_cortas))
ax.bar(x - 0.2, ndcg_sin_list, 0.4, label="Sin reranking", color="#e74c3c", alpha=0.8)
ax.bar(x + 0.2, ndcg_con_list, 0.4, label="Con reranking", color="#2ecc71", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(queries_cortas, rotation=15, ha="right")
ax.set_ylim(0, 1.2)
ax.set_ylabel("NDCG@5")
ax.set_title("Mejora de NDCG con Reranking por Cross-Encoder", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("reranking_comparativa.png", dpi=150, bbox_inches="tight")
plt.show()

# Pipeline completo
def rag_con_reranking(pregunta: str) -> str:
    docs = recuperar_con_reranking(pregunta, n_candidatos=10, n_final=3)
    contexto = "\n".join([f"- {d}" for d in docs])
    return client.messages.create(
        model=MODELO, max_tokens=300,
        messages=[{"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"}]
    ).content[0].text

print("\n=== RAG CON RERANKING ===")
print(rag_con_reranking("¿Cómo aprenden los modelos de IA de los datos?"))